# Treinamento de Modelos Preditivos - Upgrade Multianual e Otimização de Recall

## Introdução
Este notebook apresenta a versão aprimorada do modelo de predição de risco de lacunas de aprendizado. Utilizamos dados consolidados de 2020 a 2022 e implementamos técnicas avançadas de validação e otimização.

**Melhorias Implementadas:**
1. **Dados Multianuais**: Aumento da base de treinamento para maior robustez.
2. **Validação Cruzada Estratificada**: Garantia de que a performance é consistente em diferentes subconjuntos de dados.
3. **Otimização de Threshold**: Ajuste do limiar de decisão para priorizar o **Recall**, minimizando os Falsos Negativos (alunos em risco não identificados).
4. **Remoção de Data Leakage**: A variável IAN foi removida das features, mantendo a integridade do modelo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve)

import warnings
warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

# Carregamento do dataset processado multianual
df = pd.read_csv('../data/processed/processed_data.csv')
print(f'Dataset carregado com {df.shape[0]} registros.')

In [ ]:
# Definindo X e y (Removendo IAN, year e o Target)
X = df.drop(columns=['risk_gap', 'IAN', 'year'])
y = df['risk_gap']

# Divisão Treino/Teste Estratificada
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Escalonamento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Features: {X.columns.tolist()}')

## 1. Validação Cruzada Estratificada

Avaliamos a consistência dos modelos usando 3-folds.

In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    data = X_train_scaled if name == 'Logistic Regression' else X_train
    scores = cross_val_score(model, data, y_train, cv=skf, scoring='f1')
    print(f'{name} - F1-Score Médio (CV): {scores.mean():.4f} (+/- {scores.std():.4f})')

## 2. Otimização de Threshold para Recall

Ajustamos o limiar para garantir que identifiquemos o máximo de alunos em risco.

In [ ]:
# Treinar o melhor modelo (ex: Gradient Boosting)
best_model_name = 'Gradient Boosting'
model = models[best_model_name]
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# Analisar Precision-Recall Curve
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

# Escolher um threshold que garanta Recall > 0.8 (exemplo)
target_recall = 0.8
idx = np.where(recalls >= target_recall)[0][-1]
new_threshold = thresholds[idx]

print(f'Novo Threshold Otimizado: {new_threshold:.4f}')
y_pred_new = (y_proba >= new_threshold).astype(int)

## 3. Avaliação Final do Modelo Otimizado

In [ ]:
print(f'Métricas Finais ({best_model_name} com Threshold {new_threshold:.2f}):')
print(f'Accuracy: {accuracy_score(y_test, y_pred_new):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_new):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_new):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_new):.4f}')

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred_new)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Matriz de Confusão Otimizada ({best_model_name})')
plt.show()

## 4. Salvamento do Modelo Final

In [ ]:
model_data = {
    'model': model,
    'threshold': new_threshold,
    'features': X.columns.tolist()
}
with open('../models/model_risk_gap.pkl', 'wb') as f:
    pickle.dump(model_data, f)
print('Modelo salvo com sucesso em models/model_risk_gap.pkl')

## 5. Conclusão e Insights de Negócio

### Por que o Recall é Prioridade?
No contexto da Passos Mágicos, um **Falso Negativo** (não identificar um aluno em risco) é muito mais prejudicial do que um **Falso Positivo** (oferecer suporte a um aluno que talvez não precisasse tanto). Ao otimizar o Recall, garantimos que a rede de proteção da ONG alcance quem realmente precisa.

### Robustez Multianual
Ao treinar com dados de 2020 a 2022, o modelo aprendeu a ignorar flutuações anuais específicas e a focar em padrões consistentes de risco, tornando-o muito mais confiável para predições futuras.